In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# Replace with your actual file path
file_path1 = '/content/drive/My Drive/train.csv'
file_path2 = '/content/drive/My Drive/validation.csv'
file_path3 = '/content/drive/My Drive/test.csv'

train= pd.read_csv(file_path1)
val= pd.read_csv(file_path2)
test= pd.read_csv(file_path3)

In [4]:
import pandas as pd
import numpy as np
np.random.seed(42)

def sample_benign(data, n, seed=42):
    return data.sample(n=n, replace=(len(data) < n), random_state=seed)

def stratified_attack_sampling(attack_df, n_attack, min_per_attack, seed=42):
    attack_types = attack_df['label'].value_counts()
    samples = []

    for attack_type, count in attack_types.items():
        proportion = count / len(attack_df)
        n_samples = max(min(int(n_attack * proportion), count), min_per_attack)
        sampled = attack_df[attack_df['label'] == attack_type].sample(n=n_samples, random_state=seed)
        samples.append(sampled)

    attack_subset = pd.concat(samples, ignore_index=True)

    if len(attack_subset) > n_attack:
        attack_subset = attack_subset.sample(n=n_attack, random_state=seed)
    elif len(attack_subset) < n_attack:
        extra = attack_df.sample(n=n_attack - len(attack_subset), random_state=seed)
        attack_subset = pd.concat([attack_subset, extra], ignore_index=True)

    return attack_subset

def create_stratified_balanced_subset(data, n_total, benign_ratio, min_per_attack):
    benign = data[data['label'] == 'BenignTraffic']
    attack = data[data['label'] != 'BenignTraffic']

    n_benign = int(n_total * benign_ratio)
    n_attack = n_total - n_benign

    benign_subset = sample_benign(benign, n_benign)
    attack_subset = stratified_attack_sampling(attack, n_attack, min_per_attack)

    subset = pd.concat([benign_subset, attack_subset], ignore_index=True)
    return subset.sample(frac=1, random_state=42).reset_index(drop=True)

# Create subsets
train_subset = create_stratified_balanced_subset(train, n_total=100000, benign_ratio=0.9, min_per_attack=100)
val_subset   = create_stratified_balanced_subset(val,   n_total=30000,  benign_ratio=0.9, min_per_attack=30)
test_subset  = create_stratified_balanced_subset(test,  n_total=30000,  benign_ratio=0.8, min_per_attack=30)

# Save to CSV
train_subset.to_csv('train_balanced.csv', index=False)
val_subset.to_csv('val_balanced.csv', index=False)
test_subset.to_csv('test_balanced.csv', index=False)

In [5]:
"""
DATA PREPARATION FOR ZERO-DAY ATTACK DETECTION
Loads data, creates seen/unseen split, and prepares datasets
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SET ALL RANDOM SEEDS FOR REPRODUCIBILITY
# ============================================================================
SEED = 42
np.random.seed(SEED)

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    'train_file': 'train_balanced.csv',
    'val_file': 'val_balanced.csv',
    'test_file': 'test_balanced.csv',
    'random_seed': SEED,

    # ⭐ FIXED UNSEEN ATTACKS (from your best experiment)
    # These gave 89-93% unseen F1 in Experiment 1
    'unseen_attacks': [
        'ddos-rstfinflood',
        'ddos-slowloris',
        'dictionarybruteforce',
        'dos-http_flood',
        'recon-osscan',
        'vulnerabilityscan'
    ],

    'benign_labels': {'benign', 'normal', 'benigntraffic'}
}

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================
def load_and_prepare_data_with_attack_split(config):
    """Load datasets and split attacks into SEEN vs UNSEEN (Zero-Day Simulation)"""
    print("="*80)
    print("ZERO-DAY ATTACK SIMULATION: SEEN vs UNSEEN SPLIT")
    print("="*80)

    # Load data
    print("\nLoading data...")
    try:
        train_data = pd.read_csv(config['train_file'])
        val_data = pd.read_csv(config['val_file'])
        test_data = pd.read_csv(config['test_file'])
    except FileNotFoundError as e:
        print(f"❌ Error: {e}")
        raise

    # Find label column
    label_col = None
    for col in train_data.columns:
        if col.lower() == 'label':
            label_col = col
            break

    if label_col is None:
        raise ValueError("No 'label' column found!")

    print(f"Using label column: '{label_col}'")

    # Get all unique attack types (excluding benign)
    benign_labels = config['benign_labels']
    all_labels_train = train_data[label_col].values

    # Find all unique attack types
    unique_attacks = set()
    for label in all_labels_train:
        if str(label).lower() not in benign_labels:
            unique_attacks.add(str(label).lower())

    unique_attacks = sorted(list(unique_attacks))
    print(f"\n📊 Total unique attack types: {len(unique_attacks)}")

    # ⭐ USE FIXED UNSEEN ATTACKS (not random)
    unseen_attacks = set([attack.lower() for attack in config['unseen_attacks']])
    seen_attacks = set(unique_attacks) - unseen_attacks

    print(f"\n🔍 ATTACK SPLIT (FIXED UNSEEN ATTACKS):")
    print(f"  Seen attacks (for training):   {len(seen_attacks)} types")
    print(f"  Unseen attacks (zero-day):     {len(unseen_attacks)} types")
    print(f"\n  Seen attacks: {sorted(list(seen_attacks))[:5]}... (showing first 5)")
    print(f"\n  Unseen attacks:")
    for attack in sorted(list(unseen_attacks)):
        print(f"    • {attack}")

    # Helper function to categorize labels
    def categorize_label(label):
        label_str = str(label).lower()
        if label_str in benign_labels:
            return 'benign', 'benign'
        elif label_str in unseen_attacks:
            return 'attack', 'unseen'
        elif label_str in seen_attacks:
            return 'attack', 'seen'
        else:
            return 'attack', 'seen'  # Unknown attacks treated as seen

    # Process each dataset
    def process_dataset(data, dataset_name, include_unseen_in_training=False):
        """Process dataset and split into seen/unseen"""
        X = data.drop([label_col], axis=1).select_dtypes(include=[np.number]).fillna(0)
        labels_raw = data[label_col].values

        # Clip extreme values
        if np.any(np.isinf(X)) or np.any(np.abs(X) > 1e10):
            X = X.clip(-1e10, 1e10)

        # Create binary labels and attack type labels
        y_binary = []
        attack_types = []
        original_labels = []

        for label in labels_raw:
            binary_label, attack_type = categorize_label(label)
            y_binary.append(0 if binary_label == 'benign' else 1)
            attack_types.append(attack_type)
            original_labels.append(str(label).lower())

        y_binary = np.array(y_binary)
        attack_types = np.array(attack_types)
        original_labels = np.array(original_labels)

        # For training set, REMOVE unseen attacks completely
        if dataset_name == 'train' and not include_unseen_in_training:
            mask = attack_types != 'unseen'
            removed_count = sum(~mask)
            X = X[mask]
            y_binary = y_binary[mask]
            attack_types = attack_types[mask]
            original_labels = original_labels[mask]
            print(f"\n  ⚠ {dataset_name.upper()}: Removed {removed_count} unseen attack samples (ZERO-DAY SIMULATION)")

        print(f"\n  {dataset_name.upper()} distribution:")
        print(f"    Total samples: {len(y_binary)}")
        print(f"    Benign: {sum(attack_types == 'benign')} ({sum(attack_types == 'benign')/len(y_binary)*100:.1f}%)")
        print(f"    Seen attacks: {sum(attack_types == 'seen')} ({sum(attack_types == 'seen')/len(y_binary)*100:.1f}%)")
        print(f"    Unseen attacks: {sum(attack_types == 'unseen')} ({sum(attack_types == 'unseen')/len(y_binary)*100:.1f}%)")

        return X.values, y_binary, attack_types, original_labels

    # Process datasets
    X_train, y_train, train_types, train_labels = process_dataset(train_data, 'train', False)
    X_val, y_val, val_types, val_labels = process_dataset(val_data, 'val', True)
    X_test, y_test, test_types, test_labels = process_dataset(test_data, 'test', True)

    # Standardize features (fit only on training data - NO DATA LEAKAGE)
    print("\n🔒 Standardizing features (fitted on training data only)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    input_dim = X_train_scaled.shape[1]
    print(f"✓ Input dimensions: {input_dim}")

    return {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'train_types': train_types,
        'train_labels': train_labels,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'val_types': val_types,
        'val_labels': val_labels,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'test_types': test_types,
        'test_labels': test_labels,
        'input_dim': input_dim,
        'scaler': scaler,
        'seen_attacks': seen_attacks,
        'unseen_attacks': unseen_attacks
    }

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print("🚀 DATA PREPARATION FOR ZERO-DAY ATTACK DETECTION")
    print("="*80)

    # Load and prepare data
    data = load_and_prepare_data_with_attack_split(CONFIG)

    # Save prepared data
    print("\n💾 Saving prepared datasets...")
    np.save('X_train_scaled.npy', data['X_train'])
    np.save('y_train.npy', data['y_train'])
    np.save('train_types.npy', data['train_types'])

    np.save('X_val_scaled.npy', data['X_val'])
    np.save('y_val.npy', data['y_val'])
    np.save('val_types.npy', data['val_types'])

    np.save('X_test_scaled.npy', data['X_test'])
    np.save('y_test.npy', data['y_test'])
    np.save('test_types.npy', data['test_types'])

    # Save metadata
    import json
    metadata = {
        'input_dim': int(data['input_dim']),
        'train_size': int(len(data['y_train'])),
        'val_size': int(len(data['y_val'])),
        'test_size': int(len(data['y_test'])),
        'unseen_attacks': list(data['unseen_attacks']),
        'seen_attacks': list(data['seen_attacks'])
    }
    with open('dataset_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print("✓ Saved: X_train_scaled.npy, y_train.npy, train_types.npy")
    print("✓ Saved: X_val_scaled.npy, y_val.npy, val_types.npy")
    print("✓ Saved: X_test_scaled.npy, y_test.npy, test_types.npy")
    print("✓ Saved: dataset_metadata.json")

    print("\n" + "="*80)
    print("✅ DATA PREPARATION COMPLETE!")
    print("="*80)
    print("\n🎯 Next: Run model_training.py")


🚀 DATA PREPARATION FOR ZERO-DAY ATTACK DETECTION
ZERO-DAY ATTACK SIMULATION: SEEN vs UNSEEN SPLIT

Loading data...
Using label column: 'label'

📊 Total unique attack types: 33

🔍 ATTACK SPLIT (FIXED UNSEEN ATTACKS):
  Seen attacks (for training):   27 types
  Unseen attacks (zero-day):     6 types

  Seen attacks: ['backdoor_malware', 'browserhijacking', 'commandinjection', 'ddos-ack_fragmentation', 'ddos-http_flood']... (showing first 5)

  Unseen attacks:
    • ddos-rstfinflood
    • ddos-slowloris
    • dictionarybruteforce
    • dos-http_flood
    • recon-osscan
    • vulnerabilityscan

  ⚠ TRAIN: Removed 1194 unseen attack samples (ZERO-DAY SIMULATION)

  TRAIN distribution:
    Total samples: 98806
    Benign: 90000 (91.1%)
    Seen attacks: 8806 (8.9%)
    Unseen attacks: 0 (0.0%)

  VAL distribution:
    Total samples: 30000
    Benign: 27000 (90.0%)
    Seen attacks: 2646 (8.8%)
    Unseen attacks: 354 (1.2%)

  TEST distribution:
    Total samples: 30000
    Benign: 24000 (8

In [8]:
"""
MODEL TRAINING FOR ZERO-DAY ATTACK DETECTION
Trains all models and evaluates on test set
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, classification_report)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SET ALL RANDOM SEEDS FOR REPRODUCIBILITY
# ============================================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# Force TensorFlow determinism (TF 2.8+)
try:
    tf.config.experimental.enable_op_determinism()
except:
    pass

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    'epochs': 50,
    'batch_size': 256,
    'patience': 5,
    'random_seed': SEED
}

# ============================================================================
# LOAD PREPARED DATA
# ============================================================================
def load_prepared_data():
    """Load previously prepared datasets"""
    print("="*80)
    print("LOADING PREPARED DATA")
    print("="*80)

    print("\nLoading datasets...")
    X_train = np.load('X_train_scaled.npy')
    y_train = np.load('y_train.npy')
    train_types = np.load('train_types.npy')

    X_val = np.load('X_val_scaled.npy')
    y_val = np.load('y_val.npy')
    val_types = np.load('val_types.npy')

    X_test = np.load('X_test_scaled.npy')
    y_test = np.load('y_test.npy')
    test_types = np.load('test_types.npy')

    with open('dataset_metadata.json', 'r') as f:
        metadata = json.load(f)

    print(f"✓ Train: {len(y_train)} samples")
    print(f"✓ Val: {len(y_val)} samples")
    print(f"✓ Test: {len(y_test)} samples")
    print(f"✓ Input dimensions: {metadata['input_dim']}")

    return {
        'X_train': X_train, 'y_train': y_train, 'train_types': train_types,
        'X_val': X_val, 'y_val': y_val, 'val_types': val_types,
        'X_test': X_test, 'y_test': y_test, 'test_types': test_types,
        'input_dim': metadata['input_dim']
    }

# ============================================================================
# EVALUATION FUNCTIONS - ENHANCED
# ============================================================================
def evaluate_model_detailed(y_true, y_pred, attack_types, model_name, y_pred_proba=None):
    """Enhanced evaluation with more metrics"""
    results = {
        'model': model_name,

        # Overall metrics
        'overall_accuracy': accuracy_score(y_true, y_pred),
        'overall_precision': precision_score(y_true, y_pred, zero_division=0),
        'overall_recall': recall_score(y_true, y_pred, zero_division=0),
        'overall_f1': f1_score(y_true, y_pred, zero_division=0),

        # Benign detection
        'benign_accuracy': 0,
        'benign_precision': 0,
        'benign_recall': 0,
        'benign_f1': 0,
        'benign_samples': sum(attack_types == 'benign'),

        # Seen attacks
        'seen_accuracy': 0,
        'seen_precision': 0,
        'seen_recall': 0,
        'seen_f1': 0,
        'seen_samples': sum(attack_types == 'seen'),

        # Unseen attacks (Zero-Day)
        'unseen_accuracy': 0,
        'unseen_precision': 0,
        'unseen_recall': 0,
        'unseen_f1': 0,
        'unseen_samples': sum(attack_types == 'unseen'),
    }

    # AUC-ROC if probabilities available
    if y_pred_proba is not None:
        try:
            results['overall_auc'] = roc_auc_score(y_true, y_pred_proba)
        except:
            results['overall_auc'] = 0.0
    else:
        results['overall_auc'] = 0.0

    # Benign detection rate
    if results['benign_samples'] > 0:
        mask = attack_types == 'benign'
        results['benign_accuracy'] = accuracy_score(y_true[mask], y_pred[mask])
        # For benign, we want True Negatives
        tn = sum((y_true[mask] == 0) & (y_pred[mask] == 0))
        fp = sum((y_true[mask] == 0) & (y_pred[mask] == 1))
        results['benign_precision'] = tn / (tn + fp) if (tn + fp) > 0 else 0
        results['benign_recall'] = tn / results['benign_samples'] if results['benign_samples'] > 0 else 0
        results['benign_f1'] = 2 * results['benign_precision'] * results['benign_recall'] / (results['benign_precision'] + results['benign_recall']) if (results['benign_precision'] + results['benign_recall']) > 0 else 0

    # Seen attacks
    if results['seen_samples'] > 0:
        mask = attack_types == 'seen'
        results['seen_accuracy'] = accuracy_score(y_true[mask], y_pred[mask])
        results['seen_precision'] = precision_score(y_true[mask], y_pred[mask], zero_division=0)
        results['seen_recall'] = recall_score(y_true[mask], y_pred[mask], zero_division=0)
        results['seen_f1'] = f1_score(y_true[mask], y_pred[mask], zero_division=0)

    # Unseen attacks (CRITICAL METRIC)
    if results['unseen_samples'] > 0:
        mask = attack_types == 'unseen'
        results['unseen_accuracy'] = accuracy_score(y_true[mask], y_pred[mask])
        results['unseen_precision'] = precision_score(y_true[mask], y_pred[mask], zero_division=0)
        results['unseen_recall'] = recall_score(y_true[mask], y_pred[mask], zero_division=0)
        results['unseen_f1'] = f1_score(y_true[mask], y_pred[mask], zero_division=0)

    # Calculate performance gap
    results['seen_unseen_gap'] = results['seen_f1'] - results['unseen_f1']

    return results

# ============================================================================
# AUTOENCODER ARCHITECTURES - FIXED WITH SEEDS
# ============================================================================
def build_vanilla_ae(input_dim):
    """Vanilla Autoencoder with fixed seeds"""
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    inp = layers.Input(shape=(input_dim,))

    # Encoder
    x = layers.Dense(128, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(inp)
    x = layers.Dense(64, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    encoded = layers.Dense(32, activation='relu',
                          kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)

    # Decoder
    x = layers.Dense(64, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(encoded)
    x = layers.Dense(128, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    decoded = layers.Dense(input_dim, activation='linear',
                          kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)

    model = Model(inp, decoded, name='Vanilla_AE')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

    return model

def build_denoising_ae(input_dim):
    """Denoising Autoencoder with fixed seeds"""
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    inp = layers.Input(shape=(input_dim,))

    # Add noise
    x = layers.GaussianNoise(0.1, seed=SEED)(inp)

    # Encoder with dropout
    x = layers.Dense(128, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    x = layers.Dropout(0.2, seed=SEED)(x)
    x = layers.Dense(96, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    x = layers.Dropout(0.2, seed=SEED)(x)
    x = layers.Dense(64, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    encoded = layers.Dense(32, activation='relu',
                          kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)

    # Decoder
    x = layers.Dense(64, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(encoded)
    x = layers.Dense(96, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    x = layers.Dense(128, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    decoded = layers.Dense(input_dim, activation='linear',
                          kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)

    model = Model(inp, decoded, name='Denoising_AE')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

    return model

def build_supervised_ae(input_dim):
    """Supervised AE with classification head - fixed seeds"""
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    inp = layers.Input(shape=(input_dim,))

    # Encoder
    x = layers.Dense(256, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3, seed=SEED)(x)
    x = layers.Dense(128, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3, seed=SEED)(x)
    x = layers.Dense(64, activation='relu',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)
    encoded = layers.Dense(32, activation='relu',
                          kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x)

    # Classification head
    x_class = layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(encoded)
    x_class = layers.Dropout(0.2, seed=SEED)(x_class)
    classification = layers.Dense(1, activation='sigmoid', name='classification',
                                 kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x_class)

    # Reconstruction head
    x_rec = layers.Dense(64, activation='relu',
                        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(encoded)
    x_rec = layers.Dense(128, activation='relu',
                        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x_rec)
    x_rec = layers.Dense(256, activation='relu',
                        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x_rec)
    reconstruction = layers.Dense(input_dim, activation='linear', name='reconstruction',
                                 kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))(x_rec)

    model = Model(inp, [classification, reconstruction], name='Supervised_AE')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss={'classification': 'binary_crossentropy', 'reconstruction': 'mse'},
        loss_weights={'classification': 1.0, 'reconstruction': 0.5},
        metrics={'classification': ['accuracy']}
    )

    return model

# ============================================================================
# THRESHOLD FINDING
# ============================================================================
def find_optimal_threshold_f1(errors, y_true):
    """Grid search for best F1-score"""
    best_f1 = 0
    best_threshold = 0
    percentiles = np.linspace(10, 95, 50)

    for percentile in percentiles:
        threshold = np.percentile(errors, percentile)
        predictions = (errors > threshold).astype(int)
        f1 = f1_score(y_true, predictions, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_baseline_models(data):
    """Train baseline models"""
    print("\n" + "="*80)
    print("TRAINING BASELINE MODELS")
    print("="*80)

    results = []

    # Random Forest
    print("\n[1/2] Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                n_jobs=-1, random_state=SEED)
    start = time.time()
    rf.fit(data['X_train'], data['y_train'])
    train_time = time.time() - start

    rf_pred = rf.predict(data['X_test'])
    rf_pred_proba = rf.predict_proba(data['X_test'])[:, 1]

    rf_results = evaluate_model_detailed(data['y_test'], rf_pred,
                                        data['test_types'], 'Random Forest',
                                        rf_pred_proba)
    rf_results['train_time'] = train_time
    rf_results['type'] = 'Baseline'
    results.append(rf_results)

    print(f"  Overall F1: {rf_results['overall_f1']:.4f}")
    print(f"  Seen F1: {rf_results['seen_f1']:.4f}")
    print(f"  Unseen F1: {rf_results['unseen_f1']:.4f} ⭐")
    print(f"  Gap: {rf_results['seen_unseen_gap']:.4f}")

    # Logistic Regression
    print("\n[2/2] Logistic Regression...")
    lr = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)
    start = time.time()
    lr.fit(data['X_train'], data['y_train'])
    train_time = time.time() - start

    lr_pred = lr.predict(data['X_test'])
    lr_pred_proba = lr.predict_proba(data['X_test'])[:, 1]

    lr_results = evaluate_model_detailed(data['y_test'], lr_pred,
                                        data['test_types'], 'Logistic Regression',
                                        lr_pred_proba)
    lr_results['train_time'] = train_time
    lr_results['type'] = 'Baseline'
    results.append(lr_results)

    print(f"  Overall F1: {lr_results['overall_f1']:.4f}")
    print(f"  Seen F1: {lr_results['seen_f1']:.4f}")
    print(f"  Unseen F1: {lr_results['unseen_f1']:.4f} ⭐")
    print(f"  Gap: {lr_results['seen_unseen_gap']:.4f}")

    return results

def train_unsupervised_autoencoders(data, config):
    """Train unsupervised autoencoders on BENIGN data only"""
    print("\n" + "="*80)
    print("TRAINING UNSUPERVISED AUTOENCODERS (BENIGN DATA ONLY)")
    print("="*80)

    models_config = [
        ('Vanilla AE', build_vanilla_ae),
        ('Denoising AE', build_denoising_ae)
    ]

    results = []

    # Train only on benign samples
    X_train_benign = data['X_train'][data['y_train'] == 0]
    X_val_benign = data['X_val'][data['y_val'] == 0]

    print(f"Training on {len(X_train_benign)} benign samples\n")

    for idx, (model_name, build_fn) in enumerate(models_config, 1):
        print(f"[{idx}/{len(models_config)}] {model_name}")
        print("-" * 60)

        # Reset seeds before building each model
        tf.random.set_seed(SEED)
        np.random.seed(SEED)

        model = build_fn(data['input_dim'])

        start = time.time()
        model.fit(
            X_train_benign, X_train_benign,
            epochs=config['epochs'],
            batch_size=config['batch_size'],
            validation_data=(X_val_benign, X_val_benign),
            verbose=0,
            shuffle=True,  # Important for learning
            callbacks=[keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=config['patience'],
                restore_best_weights=True, verbose=1
            )]
        )
        train_time = time.time() - start

        # Find threshold on validation set
        val_recon = model.predict(data['X_val'], verbose=0)
        val_errors = np.mean(np.abs(data['X_val'] - val_recon), axis=1)
        threshold, _ = find_optimal_threshold_f1(val_errors, data['y_val'])

        # Test evaluation
        test_recon = model.predict(data['X_test'], verbose=0)
        test_errors = np.mean(np.abs(data['X_test'] - test_recon), axis=1)
        predictions = (test_errors > threshold).astype(int)

        ae_results = evaluate_model_detailed(
            data['y_test'], predictions, data['test_types'], model_name
        )
        ae_results['train_time'] = train_time
        ae_results['type'] = 'Unsupervised'
        results.append(ae_results)

        print(f"  Overall F1: {ae_results['overall_f1']:.4f}")
        print(f"  Seen F1: {ae_results['seen_f1']:.4f}")
        print(f"  Unseen F1: {ae_results['unseen_f1']:.4f} ⭐")
        print(f"  Gap: {ae_results['seen_unseen_gap']:.4f}\n")

    return results

def train_supervised_autoencoders(data, config):
    """Train supervised autoencoders"""
    print("\n" + "="*80)
    print("TRAINING SUPERVISED AUTOENCODERS")
    print("="*80)

    results = []

    print(f"Training on {len(data['y_train'])} samples (benign + seen attacks)\n")

    print("[1/1] Supervised AE")
    print("-" * 60)

    # Reset seeds
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    model = build_supervised_ae(data['input_dim'])

    start = time.time()
    model.fit(
        data['X_train'],
        {'classification': data['y_train'], 'reconstruction': data['X_train']},
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        validation_data=(
            data['X_val'],
            {'classification': data['y_val'], 'reconstruction': data['X_val']}
        ),
        verbose=0,
        shuffle=True,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=config['patience'],
            restore_best_weights=True,
            mode='min',
            verbose=1
        )]
    )
    train_time = time.time() - start

    # Test evaluation
    predictions_proba, _ = model.predict(data['X_test'], verbose=0)
    predictions = (predictions_proba > 0.5).astype(int).flatten()

    sae_results = evaluate_model_detailed(
        data['y_test'], predictions, data['test_types'], 'Supervised AE',
        predictions_proba.flatten()
    )
    sae_results['train_time'] = train_time
    sae_results['type'] = 'Supervised'
    results.append(sae_results)

    print(f"  Overall F1: {sae_results['overall_f1']:.4f}")
    print(f"  Seen F1: {sae_results['seen_f1']:.4f}")
    print(f"  Unseen F1: {sae_results['unseen_f1']:.4f} ⭐")
    print(f"  Gap: {sae_results['seen_unseen_gap']:.4f}\n")

    return results

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main training function"""
    print("\n" + "="*80)
    print("🚀 MODEL TRAINING FOR ZERO-DAY ATTACK DETECTION")
    print("="*80)

    # Load prepared data
    data = load_prepared_data()

    # Train all models
    all_results = []

    # 1. Baseline models
    baseline_results = train_baseline_models(data)
    all_results.extend(baseline_results)

    # 2. Unsupervised autoencoders
    unsupervised_results = train_unsupervised_autoencoders(data, CONFIG)
    all_results.extend(unsupervised_results)

    # 3. Supervised autoencoders
    supervised_results = train_supervised_autoencoders(data, CONFIG)
    all_results.extend(supervised_results)

    # Save results
    print("\n" + "="*80)
    print("💾 SAVING RESULTS")
    print("="*80)

    results_df = pd.DataFrame(all_results)
    results_df.to_csv('model_results.csv', index=False)
    print("✓ Saved: model_results.csv")

    # Save as pickle for result_analysis.py
    import pickle
    with open('all_results.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    print("✓ Saved: all_results.pkl")

    print("\n" + "="*80)
    print("✅ MODEL TRAINING COMPLETE!")
    print("="*80)
    print("\n🎯 Next: Run result_analysis.py")

    return all_results

if __name__ == "__main__":
    results = main()


🚀 MODEL TRAINING FOR ZERO-DAY ATTACK DETECTION
LOADING PREPARED DATA

Loading datasets...
✓ Train: 98806 samples
✓ Val: 30000 samples
✓ Test: 30000 samples
✓ Input dimensions: 46

TRAINING BASELINE MODELS

[1/2] Random Forest...
  Overall F1: 0.9733
  Seen F1: 0.9818
  Unseen F1: 0.8966 ⭐
  Gap: 0.0852

[2/2] Logistic Regression...
  Overall F1: 0.9540
  Seen F1: 0.9658
  Unseen F1: 0.8870 ⭐
  Gap: 0.0788

TRAINING UNSUPERVISED AUTOENCODERS (BENIGN DATA ONLY)
Training on 90000 benign samples

[1/2] Vanilla AE
------------------------------------------------------------
Epoch 33: early stopping
Restoring model weights from the end of the best epoch: 28.
  Overall F1: 0.9379
  Seen F1: 0.9700
  Unseen F1: 0.9153 ⭐
  Gap: 0.0548

[2/2] Denoising AE
------------------------------------------------------------
Epoch 28: early stopping
Restoring model weights from the end of the best epoch: 23.
  Overall F1: 0.9115
  Seen F1: 0.9464
  Unseen F1: 0.9207 ⭐
  Gap: 0.0257


TRAINING SUPERVISED 

In [9]:
"""
RESULT ANALYSIS AND VISUALIZATION
Prints detailed results and creates comprehensive visualizations
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")

# ============================================================================
# LOAD RESULTS
# ============================================================================
def load_results():
    """Load training results"""
    print("="*80)
    print("LOADING RESULTS")
    print("="*80)

    with open('all_results.pkl', 'rb') as f:
        results_list = pickle.load(f)

    print(f"✓ Loaded {len(results_list)} model results")
    return results_list

# ============================================================================
# DETAILED RESULTS SUMMARY WITH ENHANCED METRICS
# ============================================================================
def print_detailed_summary(results_list):
    """Print comprehensive results summary"""
    print("\n" + "="*80)
    print("📊 COMPREHENSIVE RESULTS SUMMARY")
    print("="*80)

    # Create detailed DataFrame
    summary_data = []
    for r in results_list:
        summary_data.append({
            'Model': r['model'],
            'Type': r['type'],
            'Overall F1': f"{r['overall_f1']:.4f}",
            'Overall Acc': f"{r['overall_accuracy']:.4f}",
            'Overall Prec': f"{r['overall_precision']:.4f}",
            'Overall Rec': f"{r['overall_recall']:.4f}",
            'AUC': f"{r.get('overall_auc', 0.0):.4f}",
            'Benign Acc': f"{r['benign_accuracy']:.4f}",
            'Seen F1': f"{r['seen_f1']:.4f}",
            'Seen Prec': f"{r['seen_precision']:.4f}",
            'Seen Rec': f"{r['seen_recall']:.4f}",
            'Unseen F1': f"{r['unseen_f1']:.4f}",
            'Unseen Prec': f"{r['unseen_precision']:.4f}",
            'Unseen Rec': f"{r['unseen_recall']:.4f}",
            'Gap': f"{r['seen_unseen_gap']:.4f}",
            'Train Time': f"{r['train_time']:.2f}s"
        })

    df = pd.DataFrame(summary_data)

    # Print main table
    print("\n" + "="*80)
    print("MAIN RESULTS TABLE")
    print("="*80)
    main_cols = ['Model', 'Type', 'Overall F1', 'Benign Acc', 'Seen F1',
                 'Unseen F1', 'Gap', 'Train Time']
    print("\n" + df[main_cols].to_string(index=False))

    # Print detailed metrics table
    print("\n" + "="*80)
    print("DETAILED METRICS TABLE")
    print("="*80)
    detail_cols = ['Model', 'Overall Prec', 'Overall Rec', 'AUC',
                   'Seen Prec', 'Seen Rec', 'Unseen Prec', 'Unseen Rec']
    print("\n" + df[detail_cols].to_string(index=False))

    # Key insights
    print("\n" + "="*80)
    print("🎯 KEY INSIGHTS")
    print("="*80)

    unseen_f1_scores = [r['unseen_f1'] for r in results_list]
    gaps = [r['seen_unseen_gap'] for r in results_list]

    best_unseen_idx = np.argmax(unseen_f1_scores)
    best_gap_idx = np.argmin(np.abs(gaps))

    print(f"\n1. Best Zero-Day Detection:")
    print(f"   → {results_list[best_unseen_idx]['model']}")
    print(f"   → Unseen F1: {unseen_f1_scores[best_unseen_idx]:.4f}")
    print(f"   → Unseen Precision: {results_list[best_unseen_idx]['unseen_precision']:.4f}")
    print(f"   → Unseen Recall: {results_list[best_unseen_idx]['unseen_recall']:.4f}")

    print(f"\n2. Best Generalization (Smallest Gap):")
    print(f"   → {results_list[best_gap_idx]['model']}")
    print(f"   → Gap: {gaps[best_gap_idx]:.4f}")
    print(f"   → Seen F1: {results_list[best_gap_idx]['seen_f1']:.4f}")
    print(f"   → Unseen F1: {results_list[best_gap_idx]['unseen_f1']:.4f}")

    print(f"\n3. Performance Spread:")
    print(f"   → Unseen F1 Range: {min(unseen_f1_scores):.4f} - {max(unseen_f1_scores):.4f}")
    print(f"   → Gap Range: {min(gaps):.4f} - {max(gaps):.4f}")

    print(f"\n4. Sample Statistics:")
    print(f"   → Benign samples: {results_list[0]['benign_samples']}")
    print(f"   → Seen attack samples: {results_list[0]['seen_samples']}")
    print(f"   → Unseen attack samples: {results_list[0]['unseen_samples']}")

    # Save summary
    df.to_csv('results_summary_detailed.csv', index=False)
    print("\n✅ Saved: results_summary_detailed.csv")

    return df

# ============================================================================
# COMPREHENSIVE VISUALIZATIONS
# ============================================================================
def create_comparison_visualizations(results_list):
    """Create comprehensive comparison visualizations"""
    print("\n" + "="*80)
    print("CREATING VISUALIZATIONS")
    print("="*80)

    # Prepare data
    models = [r['model'] for r in results_list]
    types = [r['type'] for r in results_list]

    overall_f1 = [r['overall_f1'] for r in results_list]
    seen_f1 = [r['seen_f1'] for r in results_list]
    unseen_f1 = [r['unseen_f1'] for r in results_list]
    benign_acc = [r['benign_accuracy'] for r in results_list]
    gaps = [r['seen_unseen_gap'] for r in results_list]

    # Colors
    type_colors = {
        'Baseline': '#e74c3c',
        'Unsupervised': '#3498db',
        'Supervised': '#2ecc71'
    }
    colors = [type_colors[t] for t in types]

    # Create figure
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # 1. Overall Performance
    ax1 = fig.add_subplot(gs[0, 0])
    x_pos = np.arange(len(models))
    ax1.bar(x_pos, overall_f1, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax1.set_ylabel('F1-Score', fontweight='bold', fontsize=11)
    ax1.set_title('Overall F1-Score', fontweight='bold', fontsize=12)
    ax1.set_ylim([0, 1.0])
    ax1.axhline(y=0.9, color='green', linestyle='--', alpha=0.3)
    ax1.grid(axis='y', alpha=0.3)

    # 2. Seen vs Unseen Comparison ⭐⭐⭐ MOST IMPORTANT
    ax2 = fig.add_subplot(gs[0, 1])
    x_pos = np.arange(len(models))
    width = 0.35
    ax2.bar(x_pos - width/2, seen_f1, width, label='Seen Attacks',
            color='#2ecc71', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax2.bar(x_pos + width/2, unseen_f1, width, label='Unseen Attacks (Zero-Day)',
            color='#e74c3c', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax2.set_ylabel('F1-Score', fontweight='bold', fontsize=11)
    ax2.set_title('🎯 SEEN vs UNSEEN Attack Detection', fontweight='bold', fontsize=12)
    ax2.set_ylim([0, 1.0])
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)

    # 3. Performance Gap ⭐⭐
    ax3 = fig.add_subplot(gs[0, 2])
    bars = ax3.bar(x_pos, gaps, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for i, (bar, gap) in enumerate(zip(bars, gaps)):
        if gap < 0.05:
            bar.set_color('#2ecc71')
        elif gap < 0.15:
            bar.set_color('#f39c12')
        else:
            bar.set_color('#e74c3c')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax3.set_ylabel('F1 Gap (Seen - Unseen)', fontweight='bold', fontsize=11)
    ax3.set_title('⚠️ Zero-Day Detection Gap (Lower = Better)', fontweight='bold', fontsize=12)
    ax3.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax3.grid(axis='y', alpha=0.3)

    # 4. Benign Detection Rate
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.bar(x_pos, benign_acc, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax4.set_ylabel('Accuracy', fontweight='bold', fontsize=11)
    ax4.set_title('Benign Traffic Detection', fontweight='bold', fontsize=12)
    ax4.set_ylim([0, 1.0])
    ax4.axhline(y=0.95, color='green', linestyle='--', alpha=0.3)
    ax4.grid(axis='y', alpha=0.3)

    # 5. Detailed Breakdown Heatmap
    ax5 = fig.add_subplot(gs[1, 1:])
    metrics_data = []
    for r in results_list:
        metrics_data.append([
            r['overall_f1'],
            r['benign_accuracy'],
            r['seen_f1'],
            r['unseen_f1'],
            r['seen_unseen_gap']
        ])
    metrics_df = pd.DataFrame(
        metrics_data,
        columns=['Overall F1', 'Benign Acc', 'Seen F1', 'Unseen F1', 'Gap (S-U)'],
        index=models
    )
    sns.heatmap(metrics_df.T, annot=True, fmt='.3f', cmap='RdYlGn',
                cbar_kws={'label': 'Score'}, ax=ax5, linewidths=1, linecolor='black')
    ax5.set_title('📊 Detailed Performance Heatmap', fontweight='bold', fontsize=12)
    ax5.set_xlabel('Models', fontweight='bold', fontsize=11)

    # 6. Seen vs Unseen Scatter
    ax6 = fig.add_subplot(gs[2, 0])
    ax6.scatter(seen_f1, unseen_f1, s=200, c=colors, alpha=0.7, edgecolors='black', linewidths=2)
    for i, model in enumerate(models):
        ax6.annotate(model, (seen_f1[i], unseen_f1[i]),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, alpha=0.8)
    ax6.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect Generalization')
    ax6.set_xlabel('Seen Attacks F1', fontweight='bold', fontsize=11)
    ax6.set_ylabel('Unseen Attacks F1', fontweight='bold', fontsize=11)
    ax6.set_title('Generalization: Seen vs Unseen', fontweight='bold', fontsize=12)
    ax6.set_xlim([0, 1.05])
    ax6.set_ylim([0, 1.05])
    ax6.legend(fontsize=9)
    ax6.grid(True, alpha=0.3)

    # 7. Training Time Comparison
    ax7 = fig.add_subplot(gs[2, 1])
    train_times = [r['train_time'] for r in results_list]
    ax7.bar(x_pos, train_times, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax7.set_xticks(x_pos)
    ax7.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax7.set_ylabel('Time (seconds)', fontweight='bold', fontsize=11)
    ax7.set_title('Training Time', fontweight='bold', fontsize=12)
    ax7.set_yscale('log')
    ax7.grid(axis='y', alpha=0.3)

    # 8. Summary Table
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.axis('off')

    summary_text = "📋 SUMMARY RANKINGS\n" + "="*40 + "\n\n"

    best_overall_idx = np.argmax(overall_f1)
    summary_text += f"🏆 Best Overall F1:\n   {models[best_overall_idx]}: {overall_f1[best_overall_idx]:.4f}\n\n"

    best_unseen_idx = np.argmax(unseen_f1)
    summary_text += f"⭐ Best Unseen F1:\n   {models[best_unseen_idx]}: {unseen_f1[best_unseen_idx]:.4f}\n\n"

    smallest_gap_idx = np.argmin(np.abs(gaps))
    summary_text += f"🎯 Best Generalization:\n   {models[smallest_gap_idx]}: {gaps[smallest_gap_idx]:.4f}\n\n"

    fastest_idx = np.argmin(train_times)
    summary_text += f"⚡ Fastest Training:\n   {models[fastest_idx]}: {train_times[fastest_idx]:.2f}s\n\n"

    summary_text += "\n" + "="*40 + "\nMODEL TYPES:\n"
    for model_type, color in type_colors.items():
        summary_text += f"  ■ {model_type}\n"

    ax8.text(0.1, 0.9, summary_text, transform=ax8.transAxes,
            fontsize=10, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

    plt.suptitle('🔍 Comprehensive Zero-Day Attack Detection Performance Analysis',
                fontsize=16, fontweight='bold', y=0.995)

    plt.savefig('comprehensive_analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: comprehensive_analysis.png")
    plt.close()

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main analysis function"""
    print("\n" + "="*80)
    print("🚀 RESULT ANALYSIS AND VISUALIZATION")
    print("="*80)

    # Load results
    results_list = load_results()

    # Print detailed summary
    df = print_detailed_summary(results_list)

    # Create visualizations
    create_comparison_visualizations(results_list)

    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    print("\n📁 Generated Files:")
    print("  - results_summary_detailed.csv")
    print("  - comprehensive_analysis.png")
    print("\n🎯 Results ready for your paper!")

if __name__ == "__main__":
    main()


🚀 RESULT ANALYSIS AND VISUALIZATION
LOADING RESULTS
✓ Loaded 5 model results

📊 COMPREHENSIVE RESULTS SUMMARY

MAIN RESULTS TABLE

              Model         Type Overall F1 Benign Acc Seen F1 Unseen F1    Gap Train Time
      Random Forest     Baseline     0.9733     1.0000  0.9818    0.8966 0.0852     28.07s
Logistic Regression     Baseline     0.9540     0.9980  0.9658    0.8870 0.0788      3.23s
         Vanilla AE Unsupervised     0.9379     0.9863  0.9700    0.9153 0.0548     73.64s
       Denoising AE Unsupervised     0.9115     0.9832  0.9464    0.9207 0.0257     98.92s
      Supervised AE   Supervised     0.9668     0.9996  0.9732    0.9180 0.0552     67.14s

DETAILED METRICS TABLE

              Model Overall Prec Overall Rec    AUC Seen Prec Seen Rec Unseen Prec Unseen Rec
      Random Forest       1.0000      0.9480 0.9983    1.0000   0.9642      1.0000     0.8125
Logistic Regression       0.9915      0.9192 0.9870    1.0000   0.9338      1.0000     0.7969
         Vanill